# CodeAlpha Task 4: Healthcare Drug Review Sentiment Analysis
Logistic Regression vs LinearSVC on the UCI Drug Review Dataset (Druglib.com). Written for Google Colab.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

print("Healthcare sentiment analysis project started!")

!pip -q install ucimlrepo

import ucimlrepo

print("✅ ucimlrepo installed and imported successfully!")

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

# Fetch the healthcare drug review dataset
drug_reviews = fetch_ucirepo(id=461)

# Separate the features and target information
X = drug_reviews.data.features
y = drug_reviews.data.targets

# Combine them into one DataFrame
df = pd.concat([X, y], axis=1)

In [ ]:
print("Dataset loaded successfully!")
print("Number of rows and columns:", df.shape)

df.head()

from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

print("COLUMN NAMES")
print(df.columns.tolist())

print("\nDATASET INFORMATION")
df.info()

In [ ]:
print("\nMISSING VALUES")
print(df.isnull().sum())

print("\nFIRST 5 REVIEWS")
df.head()

print(drug_reviews.variables[
    ["name", "role", "type", "description"]
].to_string(index=False))

print(drug_reviews.variables[
    ["name", "role", "type", "description"]
].to_string(index=False))

In [ ]:
# Combine the three written review columns
review_columns = [
    "benefitsReview",
    "sideEffectsReview",
    "commentsReview"
]

df["combined_review"] = (
    df[review_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.strip()
)

In [ ]:
print("Combined review column created!")
print("Total rows:", len(df))
print("Empty combined reviews:", (df["combined_review"] == "").sum())

df[["benefitsReview", "sideEffectsReview",
    "commentsReview", "combined_review"]].head(3)

import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
df["clean_review"] = df["combined_review"].apply(clean_text)

df[["combined_review", "clean_review"]].head(5)

print("Rating counts:")
print(df["rating"].value_counts().sort_index())

print("\nRating summary:")
print(df["rating"].describe())

df["rating"].value_counts().sort_index().plot(
    kind="bar",
    title="Distribution of Drug Review Ratings",
    xlabel="Rating (1–10)",
    ylabel="Number of Reviews",
    rot=0
)

In [ ]:
plt.tight_layout()
plt.show()

import matplotlib.pyplot as plt

def assign_sentiment(rating):
    if rating <= 4:
        return "Negative"
    elif rating <= 6:
        return "Neutral"
    else:
        return "Positive"

In [ ]:
df["sentiment"] = df["rating"].apply(assign_sentiment)

print(df["sentiment"].value_counts())
print("\nPercentages:")
print((df["sentiment"].value_counts(normalize=True) * 100).round(2))

df["sentiment"].value_counts().reindex(
    ["Negative", "Neutral", "Positive"]
).plot(
    kind="bar",
    title="Sentiment Distribution of Drug Reviews",
    xlabel="Sentiment",
    ylabel="Number of Reviews",
    rot=0
)

In [ ]:
plt.tight_layout()
plt.show()

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [ ]:
print("Machine-learning tools imported successfully!")

# Exclude rows without meaningful review text
model_df = df[df["clean_review"].str.len() > 0].copy()

X = model_df["clean_review"]
y = model_df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))
print("Total reviews used:", len(X))

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
        stop_words="english"
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

In [ ]:
model.fit(X_train, y_train)

print("Model training completed successfully!")

y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=["Negative", "Neutral", "Positive"]
)

In [ ]:
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Neutral", "Positive"]
)

display.plot(values_format="d")
plt.title("Confusion Matrix: Drug Review Sentiment Model")
plt.tight_layout()
plt.show()

In [ ]:
new_reviews = [
    "The medicine helped my symptoms and I feel much better.",
    "I experienced severe side effects and stopped taking it.",
    "I have been using this medication for two weeks."
]

cleaned_new_reviews = [clean_text(review) for review in new_reviews]
predictions = model.predict(cleaned_new_reviews)

for review, prediction in zip(new_reviews, predictions):
    print("Review:", review)
    print("Predicted sentiment:", prediction)
    print("-" * 60)

In [ ]:
results = model_df.loc[X_test.index, [
    "urlDrugName", "condition", "rating", "sentiment"
]].copy()

results["predicted_sentiment"] = y_pred
results["correct_prediction"] = (
    results["sentiment"] == results["predicted_sentiment"]
)

results.to_csv(
    "CodeAlpha_Task4_Healthcare_Sentiment_Results.csv",
    index=False
)

In [ ]:
print("Results saved!")
print("Exported rows:", len(results))

from google.colab import files

files.download("CodeAlpha_Task4_Healthcare_Sentiment_Results.csv")

from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Create a second model
svm_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
        stop_words="english",
        sublinear_tf=True
    )),
    ("classifier", LinearSVC(
        class_weight="balanced",
        random_state=42
    ))
])

In [ ]:
# Train the model
svm_model.fit(X_train, y_train)

# Make predictions
svm_pred = svm_model.predict(X_test)

# Compare both models
print("LOGISTIC REGRESSION")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Macro F1:", round(f1_score(y_test, y_pred, average="macro"), 4))

In [ ]:
print("\nLINEAR SVC")
print("Accuracy:", round(accuracy_score(y_test, svm_pred), 4))
print("Macro F1:", round(f1_score(y_test, svm_pred, average="macro"), 4))

print("\nLINEAR SVC CLASSIFICATION REPORT")
print(classification_report(y_test, svm_pred, zero_division=0))

import pandas as pd
import matplotlib.pyplot as plt

comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "LinearSVC"],
    "Accuracy": [0.6828, 0.7346],
    "Macro F1-score": [0.5296, 0.5458]
})

In [ ]:
ax = comparison.set_index("Model").plot(
    kind="bar",
    figsize=(9, 5),
    ylim=(0, 1),
    rot=0
)

plt.title("Healthcare Drug Review Sentiment: Model Comparison")
plt.ylabel("Score")
plt.xlabel("Machine Learning Model")
plt.legend(loc="upper right")
plt.tight_layout()
plt.savefig("CodeAlpha_Task4_Model_Comparison.png", dpi=300)
plt.show()

In [ ]:
print("Model comparison chart saved successfully!")

# Save test-set predictions from the selected LinearSVC model
final_results = model_df.loc[
    X_test.index,
    ["urlDrugName", "condition", "rating", "sentiment"]
].copy()

final_results["predicted_sentiment"] = svm_pred
final_results["correct_prediction"] = (
    final_results["sentiment"] == final_results["predicted_sentiment"]
)

In [ ]:
final_results.to_csv(
    "CodeAlpha_Task4_Final_Sentiment_Results.csv",
    index=False
)

print("Final results exported successfully!")
print("Rows exported:", len(final_results))
print(
    "Correct predictions:",
    final_results["correct_prediction"].sum()
)
print(
    "Incorrect predictions:",
    (~final_results["correct_prediction"]).sum()
)

In [ ]:
from google.colab import files

files.download("CodeAlpha_Task4_Final_Sentiment_Results.csv")
files.download("CodeAlpha_Task4_Model_Comparison.png")